# Chapter 2 — NB 06: variantes qualificadas (filtro do Park) + portadores, v1 × v2

**Objetivo:** montar a base do burden — o conjunto de variantes do ZNF175 que **qualifica pelo filtro do Park**, aplicado **idêntico** às duas coortes, e o **status de portador por amostra**.

**Filtro do Park (discovery):**
- **pLOF** (consequência VEP `--most_severe`): `frameshift_variant`, `stop_gained`, `stop_lost`, `start_lost`, `splice_donor_variant`, `splice_acceptor_variant`, `transcript_ablation`.
- **MAF da coorte ≤ 0,1%** (0.001).
- *(Marcamos também missense **REVEL > 0,6** à parte — a definição do Hui, para uso opcional.)*

`--most_severe` (não `--pick`) para não subnotificar pLOF. Saídas em `results/06/`.

In [1]:
# --- Setup: union + VEP(most_severe) + REVEL(dbNSFP) ---
from pathlib import Path
from io import StringIO
import subprocess
import pandas as pd, numpy as np

BASE = Path("/project/hall/analysis/hearing-loss-genomics")
R02  = BASE / "analysis/chapter_2/results/02"
R04  = BASE / "analysis/chapter_2/results/04"
R06  = BASE / "analysis/chapter_2/results/06"; R06.mkdir(parents=True, exist_ok=True)
PLINK2 = "/appl/plink2-20240804/plink2"

union = pd.read_csv(R02 / "cmp_union_all_variants.csv")

# VEP most_severe (consequence in col 'Consequence')
lines = [l for l in open(R06 / "znf175_vep_mostsevere.tab") if not l.startswith("##")]
vep = pd.read_csv(StringIO("".join(lines)), sep="\t")
vep.columns = [c.lstrip("#") for c in vep.columns]
vep = vep[["Uploaded_variation","Consequence"]].drop_duplicates("Uploaded_variation")

# REVEL from ANNOVAR dbNSFP (SNV match)
ann = pd.read_csv(R04 / "znf175_annovar.hg38_multianno.txt", sep="\t", dtype=str, na_values=".")
snv = ann[(ann["Ref"].str.len()==1) & (ann["Alt"].str.len()==1)].copy()
snv["key"] = snv["Start"].astype(str)+":"+snv["Ref"]+":"+snv["Alt"]
snv["REVEL"] = pd.to_numeric(snv["REVEL_score"], errors="coerce")

u = union.copy()
u["vep_key"] = "19:" + u["key"]
u = u.merge(vep, left_on="vep_key", right_on="Uploaded_variation", how="left")
u = u.merge(snv[["key","REVEL"]], on="key", how="left")
print("union variants:", len(u), "| with consequence:", u["Consequence"].notna().sum())

union variants: 386 | with consequence: 384


## 1. Classificar e qualificar (por coorte)
Uma variante qualifica para a coorte X se é **pLOF** e está **presente em X com MAF ≤ 0,1%**.

In [2]:
PLOF = {"frameshift_variant","stop_gained","stop_lost","start_lost",
        "splice_donor_variant","splice_acceptor_variant","transcript_ablation"}
u["is_pLOF"]     = u["Consequence"].isin(PLOF)
u["is_missense"] = u["Consequence"] == "missense_variant"
u["revel_hi"]    = u["REVEL"] > 0.6
MAF = 0.001

# Park definition (pLOF only); present in cohort = MAF not NaN
u["qual_v1_park"] = u["is_pLOF"] & u["MAF_v1"].notna() & (u["MAF_v1"] <= MAF)
u["qual_v2_park"] = u["is_pLOF"] & u["MAF_v2"].notna() & (u["MAF_v2"] <= MAF)
# Hui definition (pLOF OR deleterious missense)
u["qual_v1_hui"]  = (u["is_pLOF"] | (u["is_missense"] & u["revel_hi"])) & u["MAF_v1"].notna() & (u["MAF_v1"] <= MAF)
u["qual_v2_hui"]  = (u["is_pLOF"] | (u["is_missense"] & u["revel_hi"])) & u["MAF_v2"].notna() & (u["MAF_v2"] <= MAF)

print("=== Park filter (pLOF + MAF<=0.1%) ===")
print("qualifying v1:", int(u["qual_v1_park"].sum()), "| v2:", int(u["qual_v2_park"].sum()))
print("shared (both):", int((u["qual_v1_park"] & u["qual_v2_park"]).sum()))
print("\n=== Hui filter (pLOF + missense REVEL>0.6 + MAF<=0.1%) ===")
print("qualifying v1:", int(u["qual_v1_hui"].sum()), "| v2:", int(u["qual_v2_hui"].sum()))

print("\nPark-qualifying variants (either cohort):")
q = u[u["qual_v1_park"] | u["qual_v2_park"]]
print(q[["key","Consequence","MAF_v1","MAF_v2","qual_v1_park","qual_v2_park"]].to_string(index=False))

u.to_csv(R06 / "znf175_qualified_variants.csv", index=False)
# ID lists for plink2 (chr:pos:ref:alt)
for c in ["v1","v2"]:
    ids = ("19:" + u.loc[u[f"qual_{c}_park"], "key"])
    ids.to_csv(R06 / f"qual_{c}_park_ids.txt", index=False, header=False)

=== Park filter (pLOF + MAF<=0.1%) ===
qualifying v1: 10 | v2: 24
shared (both): 8

=== Hui filter (pLOF + missense REVEL>0.6 + MAF<=0.1%) ===
qualifying v1: 10 | v2: 28

Park-qualifying variants (either cohort):
                key        Consequence   MAF_v1   MAF_v2  qual_v1_park  qual_v2_park
      51573356:GA:G frameshift_variant      NaN 0.000011         False          True
     51573407:CAG:C frameshift_variant 0.000044 0.000034          True          True
       51573483:T:A          stop_lost      NaN 0.000011         False          True
      51581437:A:AG frameshift_variant 0.000611 0.001098          True         False
       51581490:G:T        stop_gained 0.000044 0.000011          True          True
      51581812:C:CA frameshift_variant      NaN 0.000011         False          True
      51586647:A:AT frameshift_variant      NaN 0.000011         False          True
     51586846:ATT:A frameshift_variant      NaN 0.000011         False          True
       51587087:C:A   

## 2. Status de portador por amostra (plink2 sobre os region VCFs)
Para cada coorte, extraímos as variantes qualificadas do region VCF e somamos as dosagens:
**portador = ≥1 alelo alternativo em qualquer variante qualificada**.

In [3]:
def carriers(cohort, region_vcf):
    ids = R06 / f"qual_{cohort}_park_ids.txt"
    out = R06 / f"carriers_{cohort}"
    # plink2 --export A counts A1, which defaults to REF here -> force counting the ALT allele.
    ea = R06 / f"qual_{cohort}_altallele.txt"
    with open(ea, "w") as fh:
        for vid in ("19:" + u.loc[u[f"qual_{cohort}_park"], "key"]):
            fh.write(f"{vid}\t{vid.split(':')[3]}\n")   # ID <tab> ALT
    cmd = [PLINK2, "--vcf", str(region_vcf), "--vcf-half-call", "m",
           "--extract", str(ids), "--export", "A", "--export-allele", str(ea),
           "--memory", "4000", "--out", str(out)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); raise RuntimeError("plink2 failed")
    raw = pd.read_csv(f"{out}.raw", sep=r"\s+")
    vcols = [c for c in raw.columns if c not in
             ("FID","IID","PAT","MAT","SEX","PHENOTYPE")]
    raw["alt_alleles"] = raw[vcols].sum(axis=1, skipna=True)
    raw["carrier"] = (raw["alt_alleles"] >= 1).astype(int)
    raw[["IID","alt_alleles","carrier"]].to_csv(R06 / f"carriers_{cohort}.csv", index=False)
    return raw, len(vcols)

v1_raw, n1 = carriers("v1", R02 / "v1_znf175_strict_region.vcf.gz")
v2_raw, n2 = carriers("v2", R02 / "v2_znf175_release_region.vcf.gz")

print(f"v1: {n1} qualifying variants | N={len(v1_raw):,} | carriers={int(v1_raw['carrier'].sum())} "
      f"({v1_raw['carrier'].mean()*100:.2f}%)")
print(f"v2: {n2} qualifying variants | N={len(v2_raw):,} | carriers={int(v2_raw['carrier'].sum())} "
      f"({v2_raw['carrier'].mean()*100:.2f}%)")

v1: 10 qualifying variants | N=11,451 | carriers=34 (0.30%)
v2: 24 qualifying variants | N=43,731 | carriers=72 (0.16%)


In [4]:
# --- Summary ---
nq1=int(u["qual_v1_park"].sum()); nq2=int(u["qual_v2_park"].sum())
c1=int(v1_raw["carrier"].sum()); c2=int(v2_raw["carrier"].sum())
summary = f'''# NB 06 — ZNF175 qualifying variants + carriers (Park filter, both cohorts)

Filter (Park discovery): pLOF (VEP --most_severe HIGH-impact LoF terms) + cohort MAF <= 0.1%.

## Qualifying variants
- v1 (Freeze One, 11,451): {nq1}
- v2 (Release 2020 v2.0, 43,731): {nq2}
- shared: {int((u["qual_v1_park"] & u["qual_v2_park"]).sum())}

## Carriers (>=1 qualifying alt allele)
- v1: {c1} / {len(v1_raw):,}  ({c1/len(v1_raw)*100:.2f}%)
- v2: {c2} / {len(v2_raw):,}  ({c2/len(v2_raw)*100:.2f}%)

## Reading
This is the burden INPUT, identical pipeline both cohorts. Carrier counts + qualifying-set overlap
set up the regression (NB 07: tinnitus ~ carrier + age/age2/sex/PCs, EUR/AFR-stratified + meta).
Hui-definition counts (pLOF + missense REVEL>0.6) also computed for an optional comparison pass.

## Outputs
- znf175_qualified_variants.csv, qual_v{{1,2}}_park_ids.txt, carriers_v{{1,2}}.csv
'''
(R06 / "nb06_qualifying_summary.md").write_text(summary)
print(summary)

# NB 06 — ZNF175 qualifying variants + carriers (Park filter, both cohorts)

Filter (Park discovery): pLOF (VEP --most_severe HIGH-impact LoF terms) + cohort MAF <= 0.1%.

## Qualifying variants
- v1 (Freeze One, 11,451): 10
- v2 (Release 2020 v2.0, 43,731): 24
- shared: 8

## Carriers (>=1 qualifying alt allele)
- v1: 34 / 11,451  (0.30%)
- v2: 72 / 43,731  (0.16%)

## Reading
This is the burden INPUT, identical pipeline both cohorts. Carrier counts + qualifying-set overlap
set up the regression (NB 07: tinnitus ~ carrier + age/age2/sex/PCs, EUR/AFR-stratified + meta).
Hui-definition counts (pLOF + missense REVEL>0.6) also computed for an optional comparison pass.

## Outputs
- znf175_qualified_variants.csv, qual_v{1,2}_park_ids.txt, carriers_v{1,2}.csv



Observação que já aponta para o enigma

A frequência de portadores quase caiu pela metade: 0,30% → 0,16%. O número de portadores cresceu só ~2,1× (34→72), enquanto a coorte cresceu ~3,8× (11k→44k). Isso é exatamente a mecânica de diluição / winner's curse no nível do portador: na coorte pequena, o burden se concentra em poucas variantes um pouco menos raras (10 variantes, ~3,4 portadores cada); ao crescer, entram muitos singletons ultra-raros e a taxa regride.

⚠️ Mas isso é só metade — o decisivo é se esses portadores são casos de tinnitus. Se em v1 os 34 portadores forem enriquecidos em tinnitus e em v2 os 72 não, fecha a história. É o NB 07.